## Load libraries

In [1]:
!pip install -q -U transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.0 MB/s eta 0:00:00


## Download data

In [2]:
!wget -nc -q "https://drive.google.com/uc?export=download&id=1w0_XjEl4gCrDWxFnsJHrhsVEGTiF3Ayh" -O trainingData.zip
!unzip -q trainingData.zip

## Collect drug data

In [3]:
import json
import glob
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_drug_data(directory):
    all_data = []
    file_paths = glob.glob(os.path.join(directory, "*.json"))
    for path in file_paths:
        with open(path, 'r') as f:
            try:
                item = json.load(f)
                all_data.append(item)
            except Exception as e:
                print(f"Error loading {path}: {e}")
    return all_data

drug_data = load_drug_data("/content/trainingFilesConverted/")

## Get model and tokenizer

In [4]:
model_id = "google/medgemma-4b-it" # Requires Hugging Face Hub approval

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Build in-context prompt

In [5]:
def build_in_context_prompt(target_drug, target_text, examples, num_shots=2):
    """
    Constructs a "few-shot" prompt for MedGemma.
    """
    prompt = "You are a medical informatics expert specializing in FDA label analysis.\n\n"

    # in-context training
    for i in range(min(num_shots, len(examples))):
        ex = examples[i]
        prompt += f"### EXAMPLE {i+1}\n"
        prompt += f"DRUG: {ex['drug']}\n"
        prompt += f"TEXT: {ex['text'][:800]}...\n" # Truncated for token limits
        prompt += f"INTERACTIONS: {json.dumps(ex['interactions'])}\n\n"

    # add the task
    prompt += "### TASK\n"
    prompt += f"Extract all drug-drug interactions for the drug '{target_drug}' from the following text. "
    prompt += "Return only a structured JSON list.\n\n"
    prompt += f"TEXT:\n{target_text}\n\n"
    prompt += "INTERACTIONS:"

    return prompt

## Sample it

In [6]:
def run_inference(drug_name, text, drug_data):
    # examples that don't match the drug being tested
    shots = [ex for ex in drug_data if ex['drug'] != drug_name][:2]

    full_prompt = build_in_context_prompt(drug_name, text, shots)

    # chat template format
    formatted_input = f"<start_of_turn>user\n{full_prompt}<end_of_turn>\n<start_of_turn>assistant\n"

    inputs = tokenizer(formatted_input, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.1
        )

    # Decode only the newly generated tokens
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response

# test
test_item = drug_data[0]
print(f"--- Processing Drug: {test_item['drug']} ---")

result = run_inference(test_item['drug'], test_item['text'], drug_data)
print(result)

--- Processing Drug: Ventavis ---
```json
[
  {
    "interactionType": "Pharmacodynamic interaction",
    "precipitant": "antihypertensives",
    "effect": "increased hypotensive effect"
  },
  {
    "interactionType": "Pharmacodynamic interaction",
    "precipitant": "anticoagulants",
    "effect": "increased risk of bleeding"
  }
]
```
